# 06 - Explainability and Cold-Start Scoring
Notebook wrapper for src/explainability.py and src/cold_start.py.

Runs one borrower through phase routing and SHAP-based explanation.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Using project root: {PROJECT_ROOT}')

In [ ]:
from falabella_risk.inference.cold_start import ColdStartScorer
from falabella_risk.evaluation.explainability import ExplainabilityEngine

features = pd.read_parquet('data/processed/features.parquet')
borrower_row = features.sample(1, random_state=42).iloc[0]

scorer = ColdStartScorer(
    month2_model_path=Path('models/baseline_xgb.pkl') if Path('models/baseline_xgb.pkl').exists() else None,
    month3_model_path=Path('models/hybrid_ensemble.pkl') if Path('models/hybrid_ensemble.pkl').exists() else None,
)

score_payload = scorer.score(borrower_row)
score_payload

In [ ]:
engine = ExplainabilityEngine(
    model_path=Path('models/hybrid_ensemble.pkl'),
    embeddings_path=Path('data/processed/gnn_embeddings.parquet') if Path('data/processed/gnn_embeddings.parquet').exists() else None,
    threshold=0.5,
)

explanation = engine.predict_explain(borrower_row.drop(labels=['default_flag'], errors='ignore'))
print(json.dumps(explanation, indent=2))

In [ ]:
pd.DataFrame(explanation['top_drivers'])